[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alextfkd/TorchCode/blob/master/solutions/07_batchnorm_solution.ipynb)

# ✅ Solution: BatchNorm の実装

**Batch Normalization** を **training** と **inference** 両モードで実装する。

training mode では **batch 統計量** を使い、running estimate を更新：

$$\text{BN}(x) = \gamma \cdot \frac{x - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}} + \beta$$

ここで $\mu_B$ と $\sigma_B^2$ は **batch にまたがる** mean と variance (dim=0)。

inference mode では現在の batch stats じゃなく、与えられた **running mean/var** を使う。


> 💡 **どこで使う**：CNN の各 Conv 後に挿入する正規化。学習を高速化・安定化し、暗黙の正則化も提供する。

<details>
<summary>📖 もっと詳しく</summary>

Batch 統計で正規化 → γ で scale、β で shift。

落とし穴: train / eval で挙動が違う (running stats)、batch size が小さいと不安定 (→ GroupNorm / LayerNorm が代替)、BatchNorm 直後の bias は不要 (`bias=False`)、Dropout (#17) との順序で挙動が変わる。

</details>

### Signature
```python
def my_batch_norm(
    x: torch.Tensor,
    gamma: torch.Tensor,
    beta: torch.Tensor,
    running_mean: torch.Tensor,
    running_var: torch.Tensor,
    eps: float = 1e-5,
    momentum: float = 0.1,
    training: bool = True,
) -> torch.Tensor:
    # x: (N, D) — batch の全 sample にまたがって各 feature を normalize
    # running_mean, running_var: training 中は in-place で更新、inference ではそのまま使用
```

### Rules
- `F.batch_norm`, `nn.BatchNorm1d` などは **使わない**
- batch mean/variance は `dim=0` で `unbiased=False` で計算
- running stats の更新は PyTorch 流: `running = (1 - momentum) * running + momentum * batch_stat`
- `training=False` の時は `running_mean` / `running_var` を使う
- `x`, `gamma`, `beta` についての autograd をサポート（running stats は buffer 扱いで勾配不要）

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q --force-reinstall --no-deps git+https://github.com/alextfkd/TorchCode.git')
except ImportError:
    pass


In [ ]:
import torch

In [ ]:
# ✅ SOLUTION

import torch

def my_batch_norm(
    x,
    gamma,
    beta,
    running_mean,
    running_var,
    eps=1e-5,
    momentum=0.1,
    training=True,
):
    """BatchNorm with train/eval behavior and running stats.

    - Training: use batch stats, update running_mean / running_var in-place.
    - Inference: use running_mean / running_var as-is.
    """
    if training:
        batch_mean = x.mean(dim=0)
        batch_var = x.var(dim=0, unbiased=False)

        # Update running statistics in-place. Detach to avoid tracking gradients.
        running_mean.mul_(1 - momentum).add_(momentum * batch_mean.detach())
        running_var.mul_(1 - momentum).add_(momentum * batch_var.detach())

        mean = batch_mean
        var = batch_var
    else:
        mean = running_mean
        var = running_var

    x_norm = (x - mean) / torch.sqrt(var + eps)
    return gamma * x_norm + beta

In [ ]:
# Verify
x = torch.randn(8, 4)
gamma = torch.ones(4)
beta = torch.zeros(4)

running_mean = torch.zeros(4)
running_var = torch.ones(4)

# Training behavior: normalize with batch stats and update running stats
out_train = my_batch_norm(x, gamma, beta, running_mean, running_var, training=True)
print("[Train] Column means:", out_train.mean(dim=0))
print("[Train] Column stds: ", out_train.std(dim=0))
print("Updated running_mean:", running_mean)
print("Updated running_var:", running_var)

# Inference behavior: use running_mean / running_var only
out_eval = my_batch_norm(x, gamma, beta, running_mean, running_var, training=False)
print("[Eval] Column means (using running stats):", out_eval.mean(dim=0))

In [ ]:
from torch_judge import check
check('batchnorm')